In [ ]:
# ⛔ ЗАПУСКАТЬ ПОСЛЕДНЕЙ: после подготовки, признаков, обучения роутеров и diagnostics.
# Главная проверка: одинаковое число вызовов B2 для каждого метода и mIoU на P1.
# Улучшение подтверждено только если нижняя граница ci_lo выше нуля.
import os, sys
from pathlib import Path
REPO = Path('/content/error-geometry-routing')
assert REPO.is_dir(), 'Сначала выполните ячейку «Полная подготовка entropy-baseline». '
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from core.evaluation import _miou, _miou_batch
from routing.tiles import select

cache_p1 = routing_cache('cityscapes', 'p1')
frames = list(test.groupby('image_id', sort=True).indices.items())
effects = {}
for image_id, _ in frames:
    k = cache_p1['pos'][image_id]
    effects[image_id] = tile_effects(cache_p1['gt'][k], cache_p1['b0'][k],
                                    z['tiled'][tile_pos[image_id]], tidx, n_tiles)[:2]

rng = np.random.default_rng(0)
weights = rng.multinomial(len(frames), np.full(len(frames), 1 / len(frames)), size=2000)
miou_rows = []
for budget in (1, 2, 3):
    confs = {}
    for name, score in scores.items():
        confs[name] = np.stack([
            (effects[image_id][1] + effects[image_id][0][select(score[idx], k=budget)].sum(0)).ravel()
            for image_id, idx in frames
        ])
    ref = confs['entropy']
    ref_boot = _miou_batch((weights @ ref).reshape(-1, 19, 19))
    for name, conf in confs.items():
        miou = _miou(conf.sum(0).reshape(19, 19))
        diff = _miou_batch((weights @ conf).reshape(-1, 19, 19)) - ref_boot
        miou_rows.append({'B2-патчей на кадр': budget, 'метод': name, 'mIoU': miou,
                          'Δ к entropy': miou - _miou(ref.sum(0).reshape(19, 19)),
                          'ci_lo': np.percentile(diff, 2.5),
                          'ci_hi': np.percentile(diff, 97.5)})
miou_report = pd.DataFrame(miou_rows).sort_values(['B2-патчей на кадр', 'mIoU'], ascending=[True, False])
miou_report.to_csv(OUT / 'p1_miou_bootstrap.csv', index=False)
miou_report

# Post-hoc B0 → B2 router

Ноутбук воспроизводит второй способ из Narasimhan et al. (NeurIPS 2022) для уже замороженных SegFormer-B0 и SegFormer-B2. Он **не обучает сегментационные модели**: сначала запускает их инференс на Cityscapes val, строит таблицу патчей с softmax-энтропией, затем обучает только логистический роутер на P0 и проверяет его на P1.

Для baseline достаточно открытой папки с `val.zip` и `gtFine_trainvaltest.zip`: подготовительная ячейка ниже строит `tile_features_entropy_t256_m64.parquet` сама. Колонки `b0_error` и `b2_error` используются только как метки при обучении, а не как вход роутера на P1.

In [ ]:
# Colab: подключаем Google Drive и ставим лёгкие зависимости.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
# Не обновляем pandas/pyarrow/sklearn: они уже согласованы с образом Colab.
%pip -q install --upgrade-strategy only-if-needed gdown

In [ ]:
from pathlib import Path
# Берём только val и разметку. test.zip не нужен: у Cityscapes test нет GT.
DOWNLOAD_DIR = Path('/content/frozen-router-data')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
FILES = {
    'val.zip': '1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ',
    'gtFine_trainvaltest.zip': '1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk',
}

import gdown
for name, file_id in FILES.items():
    path = DOWNLOAD_DIR / name
    if not path.exists():
        gdown.download(id=file_id, output=str(path), quiet=False)
    assert path.exists() and path.stat().st_size > 0, f'Не удалось скачать {name}'
    print(f'{name}: {path.stat().st_size / 2**20:.1f} MiB')



In [ ]:
# Полная подготовка entropy-baseline из исходных архивов. Нужен GPU Colab;
# первый запуск занимает заметное время, но следующие используют results/.
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

# Ячейка самодостаточна: её можно запускать после перезапуска Colab,
# даже если ячейка отдельного скачивания выше не запускалась.
DOWNLOAD_DIR = Path('/content/frozen-router-data')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
FILES = {
    'val.zip': '1aErjLEt25oJEX48W-zxHwITEvNAeKxGJ',
    'gtFine_trainvaltest.zip': '1AxGkcacU_SZXPSjxUtzKgT3XSZM032bk',
}
try:
    import gdown
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    import gdown
for archive_name, file_id in FILES.items():
    archive = DOWNLOAD_DIR / archive_name
    if not archive.exists() or archive.stat().st_size == 0:
        print('Скачиваю', archive_name)
        gdown.download(id=file_id, output=str(archive), quiet=False)
    assert archive.exists() and archive.stat().st_size > 0, f'Не удалось получить {archive_name}'


assert __import__('torch').cuda.is_available(), 'В Colab включите Runtime → Change runtime type → T4 GPU.'
REPO = Path('/content/error-geometry-routing')
DRIVE_DIR = Path('/content/drive/MyDrive/error-geometry-routing')
PERSIST_DIR = DRIVE_DIR / 'posthoc_router_20261027'
PERSIST_RESULTS = PERSIST_DIR / 'results'
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch',
                    'experiments/frozen-router-results-20261025',
                    'https://github.com/Ottobiss/error-geometry-routing.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'cityscapesscripts'], check=True)
try:
    import pyarrow  # есть в стандартной среде Colab; не обновляем его без нужды
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow<24'], check=True)

# Восстанавливаем законченные этапы предыдущей сессии до любых проверок exists().
if PERSIST_RESULTS.exists():
    shutil.copytree(PERSIST_RESULTS, REPO / 'results', dirs_exist_ok=True)
    print('Восстановлены результаты с Google Drive:', PERSIST_RESULTS)

def checkpoint(stage):
    source = REPO / 'results'
    if source.exists():
        shutil.copytree(source, PERSIST_RESULTS, dirs_exist_ok=True)
        print(f'✓ Сохранено на Drive после этапа: {stage}')

checkpoint('уже рассчитанные артефакты')

DATA_ROOT = REPO / 'data' / 'cityscapes'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for archive_name in ('val.zip', 'gtFine_trainvaltest.zip'):
    archive = DOWNLOAD_DIR / archive_name
    try:
        with zipfile.ZipFile(archive) as zf:
            bad = zf.testzip()
            if bad is not None:
                raise RuntimeError(f'повреждён файл внутри архива: {bad}')
            # Уже существующие файлы пропускаем: повторный запуск не тратит место и время.
            for member in zf.infolist():
                target = DATA_ROOT / member.filename
                if not target.exists():
                    zf.extract(member, DATA_ROOT)
    except (zipfile.BadZipFile, RuntimeError) as exc:
        raise RuntimeError(f'{archive_name} не является целым ZIP-архивом: {exc}. ' +
                           'Удалите этот файл, заново выполните ячейку скачивания и повторите.') from exc
# val.zip может содержать val/<город>/..., а проект ожидает leftImg8bit/val/<город>/... .
canonical_images = DATA_ROOT / 'leftImg8bit' / 'val'
images = list(canonical_images.rglob('*_leftImg8bit.png'))
if not images:
    found = list(DATA_ROOT.rglob('*_leftImg8bit.png'))
    for image in found:
        destination = canonical_images / image.parent.name / image.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.move(str(image), str(destination))
    images = list(canonical_images.rglob('*_leftImg8bit.png'))
labels = list((DATA_ROOT / 'gtFine' / 'val').rglob('*_gtFine_labelIds.png'))
assert len(images) == len(labels) == 500, (len(images), len(labels), DATA_ROOT)

def run_module(*args):
    print('▶', ' '.join(args))
    subprocess.run([sys.executable, '-m', *args], cwd=REPO, check=True)

# B0/B2 остаются замороженными: здесь выполняется только их инференс.
if not (REPO / 'results/cityscapes/components.parquet').exists():
    run_module('features.components', '--dataset', 'cityscapes')
    checkpoint('components')
if not (REPO / 'results/cityscapes/logits_b0.npy').exists():
    run_module('features.logits', '--dataset', 'cityscapes')
    checkpoint('logits_b0')
# Кеш tiles состоит из пары .npz + .json; один .npz недостаточен.
tiles_npz = REPO / 'results/cityscapes/tiles_image_t256_m64.npz'
tiles_json = REPO / 'results/cityscapes/tiles_image_t256_m64.json'
if not (tiles_npz.exists() and tiles_json.exists()):
    run_module('features.tiles', '--dataset', 'cityscapes', '--input', 'image')
    checkpoint('b2_tiles')
print('Подготовлены B0/B2, логиты B0 и B2 на патчах.')

In [ ]:
# Строим признаки, доступные сразу после B0: энтропия, margin, границы и форма патча.
# P1 здесь используется только для будущей проверки, не для обучения.
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import numpy as np
import pandas as pd
from core.components import IGNORE, NC
from core.data import DATASETS
from core.evaluation import entropy_score, routing_cache
from core.paths import DatasetResults
from core.tiles import grid, tile_effects, tile_index
from features.logits import LogitStore

res = DatasetResults('cityscapes')
cfg = 'image_t256_m64'
tile_info = __import__('json').loads(res.tiles(cfg, 'json').read_text())
size, tile = DATASETS['cityscapes'].size, tile_info['tile']
boxes, tidx = grid(size, tile), tile_index(size, tile)
n_tiles = len(boxes)
area = np.bincount(tidx.ravel(), minlength=n_tiles)
z = np.load(res.tiles(cfg))
tile_pos = {image_id: k for k, image_id in enumerate(z['image_id'])}
logits = LogitStore('cityscapes')
rows = []
for pop in ('p0', 'p1'):
    cache = routing_cache('cityscapes', pop)
    for image_id in cache['image_id']:
        k = cache['pos'][image_id]
        gt, b0 = cache['gt'][k], cache['b0'][k]
        b2 = z['tiled'][tile_pos[image_id]]
        valid = gt != IGNORE
        _, _, delta = tile_effects(gt, b0, b2, tidx, n_tiles)
        prob = logits[image_id].softmax(0)
        ent = entropy_score(prob).cpu().numpy()
        top2 = prob.topk(2, dim=0).values.cpu().numpy()
        def tile_mean(x):
            return np.bincount(tidx.ravel(), weights=x.ravel(), minlength=n_tiles) / area
        edge = np.zeros(b0.shape, dtype=bool)
        edge[:, :-1] |= b0[:, :-1] != b0[:, 1:]
        edge[:-1, :] |= b0[:-1, :] != b0[1:, :]
        overlap = np.bincount(cache['comp'][k].ravel().astype(np.int64) * n_tiles + tidx.ravel(),
                              minlength=(cache['comp'][k].max() + 1) * n_tiles).reshape(-1, n_tiles)
        cls = np.bincount(tidx.ravel() * NC + b0.ravel(), minlength=n_tiles * NC).reshape(n_tiles, NC)
        fields = {
            'm_entropy': tile_mean(ent), 'ent_mean': tile_mean(ent),
            'ent_p90': np.array([np.quantile(ent[y0:y1, x0:x1], 0.9) for x0, y0, x1, y1 in boxes]),
            'ent_frac': tile_mean(ent > 0.5),
            'margin_mean': tile_mean(1 - (top2[0] - top2[1])),
            'pmax_mean': tile_mean(top2[0]), 'boundary': tile_mean(edge),
            'n_comp': (overlap > 0).sum(0),
        }
        fields.update({f'cls_{c}': cls[:, c] / area for c in range(NC)})
        rows.append(pd.DataFrame({
            'image_id': image_id, 'population': pop, 'tile': np.arange(n_tiles),
            **fields, 'delta': delta,
            'n_valid': np.bincount(tidx.ravel(), weights=valid.ravel(), minlength=n_tiles),
            'b0_error': np.bincount(tidx.ravel(), weights=(valid & (b0 != gt)).ravel(), minlength=n_tiles),
            'b2_error': np.bincount(tidx.ravel(), weights=(valid & (b2 != gt)).ravel(), minlength=n_tiles),
        }))
TABLE = REPO / 'results/cityscapes/tile_features_entropy_t256_m64.parquet'
pd.concat(rows, ignore_index=True).to_parquet(TABLE, index=False)
checkpoint('таблица признаков и меток')
print('Таблица готова:', TABLE)

In [ ]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/error-geometry-routing')
# TABLE создана предыдущей ячейкой из val.zip + gtFine_trainvaltest.zip.
# При желании сохраните её в Drive, чтобы следующий запуск не считал инференс заново.
TABLE = REPO / 'results/cityscapes/tile_features_entropy_t256_m64.parquet'
OUT = DRIVE_DIR / 'posthoc_router_20261026'
OUT.mkdir(parents=True, exist_ok=True)
assert TABLE.exists(), 'Сначала выполните две подготовительные ячейки выше.'
print('Таблица:', TABLE)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

class PostHocRejector:
    # Eq. (14) from Narasimhan et al.:
    # e0 * softplus(-r(x)) + (e2 + c*n) * softplus(r(x)).
    # A positive r means that B2 should be selected.
    def __init__(self, defer_cost=0.0, C=1.0):
        self.defer_cost, self.C = defer_cost, C

    def fit(self, frame, fields):
        d = frame.loc[frame.n_valid > 0].copy()
        required = set(fields) | {'b0_error', 'b2_error', 'n_valid'}
        missing = required - set(d.columns)
        if missing:
            raise ValueError('В parquet отсутствуют: ' + ', '.join(sorted(missing)) +
                             '. Пересоберите его обновлённым features.tile_features.')
        x = d[fields].to_numpy(float)
        if not np.isfinite(x).all():
            raise ValueError('В признаках есть NaN/inf')
        c_model = d.b0_error.to_numpy(float)
        c_expert = d.b2_error.to_numpy(float) + self.defer_cost * d.n_valid.to_numpy(float)
        X = np.vstack([x, x])
        y = np.r_[np.ones(len(d), dtype=int), np.zeros(len(d), dtype=int)]
        w = np.r_[c_model, c_expert]
        keep = w > 0
        self.model = make_pipeline(StandardScaler(), LogisticRegression(C=self.C, max_iter=1000, random_state=0))
        self.model.fit(X[keep], y[keep], logisticregression__sample_weight=w[keep])
        return self

    def score(self, frame, fields):
        return self.model.decision_function(frame[fields].to_numpy(float))

def gain_regression(train, test, fields):
    # E4-подход из ветки baseline-experiments: учим ожидаемый выигрыш B2 на патче.
    d = train.loc[train.n_valid > 0]
    model = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300,
                                      max_leaf_nodes=15, min_samples_leaf=50, random_state=0)
    model.fit(d[fields], d.delta / d.n_valid, sample_weight=d.n_valid)
    # Бустинг даёт одинаковые значения в листьях; энтропия устойчиво разбивает ничьи.
    return model.predict(test[fields]) + 1e-9 * test.m_entropy.to_numpy()

# Все эти признаки вычисляются из B0, до запуска B2 на конкретном патче.
UNCERTAINTY = ['ent_mean', 'ent_p90', 'ent_frac', 'margin_mean', 'pmax_mean',
               'boundary', 'n_comp'] + [f'cls_{c}' for c in range(19)]
GROUPS = {'энтропия': ['m_entropy'], 'B0-контекст': ['m_entropy'] + UNCERTAINTY}

In [ ]:
table = pd.read_parquet(TABLE)
train = table.query("population == 'p0'").copy()
test = table.query("population == 'p1'").copy()
assert len(train) and len(test), 'Ожидаются строки Cityscapes P0 и P1 в одном parquet.'
assert set(train.image_id).isdisjoint(test.image_id), 'P0 и P1 пересекаются.'
print(f'P0: {len(train)} патчей, {train.image_id.nunique()} кадров')
print(f'P1: {len(test)} патчей, {test.image_id.nunique()} кадров')

scores = {'entropy': test.m_entropy.to_numpy()}
for name, fields in GROUPS.items():
    missing = set(fields) - set(table.columns)
    if missing:
        print(f'Пропуск {name}: отсутствуют {sorted(missing)}')
        continue
    scores[f'posthoc::{name}'] = PostHocRejector(defer_cost=0.0).fit(train, fields).score(test, fields)
    scores[f'gain-gbr::{name}'] = gain_regression(train, test, fields)

out = test[['image_id', 'tile', 'population', 'delta', 'n_valid']].copy()
for name, value in scores.items():
    out[name] = value
out.to_parquet(OUT / 'p1_posthoc_scores.parquet', index=False)
out.head()

In [ ]:
# Safe residual routing: добавляем к энтропии только поправку, прошедшую
# групповую cross-validation на P0. P1 здесь не используется для выбора alpha.
from scipy.stats import rankdata
from sklearn.model_selection import GroupKFold

BLEND_FIELDS = GROUPS['B0-контекст']
BUDGETS = (1, 2, 3)
ALPHAS = np.array([0.0, 0.025, 0.05, 0.10, 0.20, 0.35, 0.50])

def regularized_gain_model(frame):
    # Намеренно проще прежнего GBR: уменьшаем риск запомнить P0.
    d = frame.loc[frame.n_valid > 0]
    model = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=120,
        max_leaf_nodes=7, min_samples_leaf=200, l2_regularization=1.0, random_state=0)
    model.fit(d[BLEND_FIELDS], d.delta / d.n_valid, sample_weight=d.n_valid)
    return model

def rank_in_frame(frame, values):
    result = np.empty(len(frame), dtype=float)
    for _, pos in frame.groupby('image_id', sort=False).indices.items():
        result[pos] = rankdata(values[pos], method='average') / len(pos)
    return result

def per_frame_gain(frame, values):
    # Независимая от mIoU proxy: сколько ошибок в среднем исправляет B2
    # при том же бюджете 1, 2 или 3 патча на кадр.
    answer = []
    for _, pos in frame.groupby('image_id', sort=False).indices.items():
        delta = frame.iloc[pos].delta.to_numpy()
        answer.append(np.mean([delta[np.argsort(values[pos])[-k:]].sum() for k in BUDGETS]))
    return np.asarray(answer)

safe_train = train.reset_index(drop=True).copy()
safe_test = test.reset_index(drop=True).copy()
oof_gain = np.empty(len(safe_train))
n_folds = min(5, safe_train.image_id.nunique())
for fit_pos, val_pos in GroupKFold(n_splits=n_folds).split(safe_train, groups=safe_train.image_id):
    model = regularized_gain_model(safe_train.iloc[fit_pos])
    oof_gain[val_pos] = model.predict(safe_train.iloc[val_pos][BLEND_FIELDS])

entropy_rank_oof = rank_in_frame(safe_train, safe_train.m_entropy.to_numpy())
gain_rank_oof = rank_in_frame(safe_train, oof_gain)
base_gain = per_frame_gain(safe_train, safe_train.m_entropy.to_numpy())
selection_rows = []
for alpha in ALPHAS:
    blend = (1 - alpha) * entropy_rank_oof + alpha * gain_rank_oof
    difference = per_frame_gain(safe_train, blend) - base_gain
    mean = difference.mean()
    se = difference.std(ddof=1) / np.sqrt(len(difference))
    selection_rows.append({'alpha': alpha, 'oof_Δ_исправленных_пикселей': mean,
                           'oof_se': se, 'passes_95pct_guard': alpha > 0 and mean > 1.96 * se})
selection = pd.DataFrame(selection_rows)
eligible = selection.query('passes_95pct_guard')
alpha = 0.0 if eligible.empty else float(eligible.sort_values('oof_Δ_исправленных_пикселей', ascending=False).iloc[0].alpha)

# Только теперь обучаемся на всех P0 и применяем выбранную поправку к P1.
full_gain = regularized_gain_model(safe_train).predict(safe_test[BLEND_FIELDS])
safe_score = ((1 - alpha) * rank_in_frame(safe_test, safe_test.m_entropy.to_numpy()) +
              alpha * rank_in_frame(safe_test, full_gain))
safe_name = f'safe-blend::B0-контекст::alpha={alpha:g}'
scores[safe_name] = safe_score
out[safe_name] = safe_score
out.to_parquet(OUT / 'p1_posthoc_scores.parquet', index=False)
selection.to_csv(OUT / 'p0_safe_blend_selection.csv', index=False)
print(f'Выбран alpha={alpha:g}; alpha=0 означает: оставить чистую entropy.')
selection


In [ ]:
# Честная диагностика ранжирования на P1. Это ещё не mIoU: для mIoU нужны
# маски B0, ответы B2 по окнам и existing routing.tile_router.
target = (out.delta > 0).to_numpy()
density = out.delta / out.n_valid.clip(lower=1)
rows = []
for name in scores:
    s = out[name].to_numpy()
    rows.append({'score': name,
                 'AUROC(delta>0)': roc_auc_score(target, s),
                 'AP(delta>0)': average_precision_score(target, s),
                 'Spearman(delta/n_valid)': spearmanr(s, density).statistic})
report = pd.DataFrame(rows).sort_values('Spearman(delta/n_valid)', ascending=False)
report.to_csv(OUT / 'p1_posthoc_diagnostics.csv', index=False)
report

## Как интерпретировать проверку

Последняя ячейка сохраняет `p1_miou_bootstrap.csv`: каждый метод получает одинаковое число патчей B2 на кадр. Улучшение над entropy подтверждается только в строке, где `Δ к entropy > 0` и весь 95%-интервал `[ci_lo, ci_hi]` лежит выше нуля. Если интервал пересекает ноль, различие не доказано.